# ProtoCloud Tutorial

ProtoCloud is a prototype-based Variational Autoencoder (VAE) for single-cell RNA-seq cell type classification. Each cell type is represented by a set of learnable **prototypes** in a shared latent space. Classification is based on similarity between a cell's latent representation and the prototypes. Gene-level interpretability is provided via **Prototype Relevance Propagation (PRP)**.

This notebook walks through the Python API using the PBMC 10K dataset.

## 1. Installation

**Input data format requirements:**

| Requirement | Location | Notes |
|-------------|----------|-------|
| Cell type labels | `adata.obs["celltype"]` | Required for labeled training |
| Gene names | `adata.var["gene_name"]` | Must be a column in `adata.var` |
| Raw counts | `adata.layers["counts"]` or `adata.X` | Use `raw=1` when raw counts are in `adata.X` |

In [1]:
!conda env create -f requirements.yml
!conda activate protocloud
!pip install -e .

Retrieving notices: ...working... done



EnvironmentFileNotFound: 'd:\DingLab\Projects\protoCloud_project\code\ProtoCloud\tutorial\requirements.yml' file not found


CondaError: Run 'conda init' before 'conda activate'



Obtaining file:///D:/DingLab/Projects/protoCloud_project/code/ProtoCloud/tutorial


ERROR: file:///D:/DingLab/Projects/protoCloud_project/code/ProtoCloud/tutorial does not appear to be a Python project: neither 'setup.py' nor 'pyproject.toml' found.


In [22]:
# !conda env create -f requirements.yml
# !conda activate protocloud
!pip install -e .

Obtaining file:///D:/DingLab/Projects/protoCloud_project/code/ProtoCloud
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Checking if build backend supports build_editable: started
  Checking if build backend supports build_editable: finished with status 'done'
  Getting requirements to build editable: started
  Getting requirements to build editable: finished with status 'done'
  Preparing editable metadata (pyproject.toml): started
  Preparing editable metadata (pyproject.toml): finished with status 'done'
  Building editable for ProtoCloud (pyproject.toml): started
  Building editable for ProtoCloud (pyproject.toml): finished with status 'done'
  Created wheel for ProtoCloud: filename=protocloud-1.0.0-0.editable-py3-none-any.whl size=4650 sha256=fc4391f4499874dd38cff13da52458c6522abab3fd95c05c87bbfc4c418c3f51
  Stored in directory: C:\Users\yun\AppData\Local\Temp\pip-ephem-wheel-cache-ml7you_3\wheels\cf\3d\e4\6b7b22cc3da40a0f32f

In [3]:
import os, shutil
import numpy as np
import pandas as pd
import torch
import ProtoCloud

# Ensure CWD is the repo root (notebook kernels start in the notebook's directory)
if os.path.basename(os.getcwd()) == 'tutorial':
    os.chdir('..')
print(f'Working directory: {os.getcwd()}')

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')

# Paths
DATA_DIR    = './tutorial/'
MODEL_DIR   = './tutorial/saved_models/PBMC_10K_full/'
RESULTS_DIR = './tutorial/results/PBMC_10K_full/'
EXP_CODE    = 'tutorial_run'

ProtoCloud.utils.makedir(MODEL_DIR)
ProtoCloud.utils.makedir(RESULTS_DIR)

c:\Users\yun\.conda\envs\protocloud\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Working directory: d:\DingLab\Projects\protoCloud_project\code\ProtoCloud
Using device: cpu


---
## Quick Start with the High-Level API

`ProtoCloudModel` wraps the full ProtoCloud pipeline — train/test split, rare-cell-type oversampling, two-step curriculum training, similarity calibration, class thresholds, save/load, gene alignment, and PRP gene-relevance explanations — into a few lines.

**Most users should start here.** For custom calibration metrics, direct prototype inspection, advanced PRP usage (per-prototype inspection, LRP explanations, custom rule maps), or fine-grained control over the training loop, continue to Section 2 and beyond for the low-level walkthrough.

This Quick Start mirrors `api_call.py` in the repository root.

### Load data

In [ ]:
import scanpy as sc
import ProtoCloud as pc

adata = sc.read(os.path.join(DATA_DIR, 'PBMC_10K_full.h5ad'))
adata

### Train a model

`ProtoCloudModel()` accepts the same architecture hyperparameters as the low-level `protoCloud` class (`latent_dim`, `num_prototypes_per_class`, `encoder_layer_sizes`, ...). `fit_model` internally handles the train/test split, rare-type augmentation, two-step training, and post-training calibration.

We use `epochs=5` here for tutorial speed — production runs typically use 100+.

In [ ]:
model = pc.ProtoCloudModel()
model.fit_model(adata, epochs=5)

### Save the trained model

`save_model` writes the network weights, `model_dict`, cell encoder, gene names, per-class certainty thresholds, and similarity calibrator to a single directory.

In [ ]:
QUICKSTART_DIR = './tutorial/saved_models/quickstart/'
model.save_model(QUICKSTART_DIR)

### Load and predict on new data

`ProtoCloudModel.load(...)` restores the full model, and `predict_model(adata)` writes predictions directly into `adata.obs`:

| Column | Meaning |
|---|---|
| `pc_prediction` | Predicted cell type label |
| `pc_sim_score` | Raw cosine similarity to the nearest prototype |
| `pc_certainty` | `'certain'` / `'ambiguous'` from per-class thresholds |
| `pc_calibrated_certainty` | Isotonic-calibrated probability |
| `pc_log_likelihood` | Log-likelihood (only when `obs_dist='nb'`) |

Gene alignment between the new dataset and the trained model's gene space is handled automatically.

In [ ]:
model = pc.ProtoCloudModel.load(QUICKSTART_DIR)
new_adata = sc.read(os.path.join(DATA_DIR, 'PBMC_10K_full.h5ad'))
model.predict_model(new_adata)
new_adata.obs[['pc_prediction', 'pc_certainty', 'pc_sim_score']].head()

### Continue training

To fine-tune a saved model on additional data, load it and call `fit_model` again. Two-step training is automatically disabled and gene alignment happens for you.

In [ ]:
model = pc.ProtoCloudModel.load(QUICKSTART_DIR)
model.fit_model(new_adata, epochs=3)
model.save_model('./tutorial/saved_models/quickstart_v2/')

---
## 2. Data Loading & Preprocessing

`scRNAData` loads an `.h5ad` file, optionally selects HVGs, and prepares the data for training.

In [4]:
# Load dataset — .h5ad file must be at <data_dir>/<dataset_name>.h5ad
data = ProtoCloud.data.scRNAData(
    dataset_name='PBMC_10K_full',
    data_dir=DATA_DIR,
)

Loading outside dataset, this will not process the data
AnnData object with n_obs × n_vars = 11527 × 3267
    obs: 'n_counts', 'batch', 'labels', 'str_labels', 'celltype'
    var: 'gene_symbols', 'n_counts-0', 'n_counts-1', 'n_counts', 'gene_name'
    uns: 'cell_types', 'log1p'
    obsm: 'design', 'normalized_qc', 'qc_pc', 'raw_qc'
    layers: 'counts'


In [5]:
# Split into train and test sets
train_idx, test_idx = data.get_split_idx(test_ratio=0.2)
train_X, test_X, train_Y, test_Y = data.split_data(train_idx, test_idx)
# train_X: np.ndarray (n_train, n_genes)  — gene expression matrix
# train_Y: np.ndarray (n_train,)          — integer-encoded cell type labels

print(f'train_X: {train_X.shape},  test_X: {test_X.shape}')

	Using layer 'counts' as input
	Augmenting rare cell types
Rare cell types: 4
['B cells'] 0.124
['CD14+ Monocytes'] 0.168
['CD4 T cells'] 0.380
['CD8 T cells'] 0.109
['Dendritic Cells'] 0.055
['FCGR3A+ Monocytes'] 0.055
['Megakaryocytes'] 0.055
['NK cells'] 0.055
train_X: (10516, 3267),  test_X: (2306, 3267)


---
## 3. Model Setup & Training

In [6]:
num_classes = len(data.cell_encoder.classes_)

model_dict = {
    'input_dim':   len(data.gene_names),
    'num_classes': num_classes,
    # 'latent_dim': 20,                 # default
    # 'num_prototypes_per_class': 6,    # default: number of prototypes per cell type
}

model = ProtoCloud.protoCloud(**model_dict).to(device)

In [7]:
# Encode test labels to integers for validation
test_y = torch.LongTensor(data.cell_encoder.transform(test_Y))

result_trend = ProtoCloud.model.run_model(
    model, train_X, train_Y,
    test_X=torch.Tensor(test_X),
    test_Y=test_y,
    validate_model=True,  # set True to compute validation accuracy every 10 epoch
    # epochs=100,            # default
    # batch_size=None,       # if None, use 128 (<100k cells) and 1024 for large datasets 
    # lr=1e-3,            # default
    # two_step=True,      # default: two-stage curriculum learning
    # recon_coef=10,      # default
    # kl_coef=2,          # default
    # stage1_ortho_coef=0,  # default: optional to add regularization in stage 1
    # ortho_coef=0.3,     # default
    # atomic_coef=1,      # default
)
# result_trend: tuple of (train_loss_list, train_acc_list, valid_acc_list)

loss coef: {'crs_ent': 1, 'recon': 10, 'kl': 2, 'ortho': 0.0, 'atomic': 0.0}
Start training
Train epoch: 0 	accu: 54.19360973754279% 	loss: 48.13958228506693 	recons: 4.215878158080868 	KL: 2.0364192244483204 	cross ent: 1.9079620489260045 	ortho: 22.7449355706936 	atomic: 0.01900434648481811
Valid epoch: 0 	accu: 60.928013876843025%
Train epoch: 10 	accu: 90.13883605933816% 	loss: 41.1024255054753 	recons: 3.9342585133343206 	KL: 0.6263106341769056 	cross ent: 0.5072193305666853 	ortho: 5.253659591442201 	atomic: -0.1001528247100551
Valid epoch: 10 	accu: 88.20468343451864%
Train epoch: 20 	accu: 96.52909851654621% 	loss: 39.49214898086176 	recons: 3.879276115719865 	KL: 0.24706474637112966 	cross ent: 0.20525815119830573 	ortho: 2.0643551175187276 	atomic: -0.19552469580638698
Valid epoch: 20 	accu: 94.23243712055508%
Updated loss coef: {'crs_ent': 1, 'recon': 10, 'kl': 2, 'ortho': 0.3, 'atomic': 1}
Train epoch: 30 	accu: 98.65918600228224% 	loss: 38.849188548762626 	recons: 3.846284

In [8]:
# Save model and associated metadata
ProtoCloud.utils.data_info_saver(data.cell_encoder, MODEL_DIR, 'cell_encoder')
ProtoCloud.utils.data_info_saver(data.gene_names, MODEL_DIR, 'gene_names')
ProtoCloud.model.save_model(model, model_dict, MODEL_DIR, EXP_CODE, EXP_CODE)

	Saving model to ./tutorial/saved_models/PBMC_10K_full/tutorial_run.pth...
Model saved
model dict saved


---
## 4. Predictions & Confidence Scores

### 4a. Training-set predictions — for thresholds & calibration

In [9]:
# Get predictions on training data to learn per-class certainty thresholds
train_pred = pd.DataFrame(ProtoCloud.model.get_predictions(model, torch.Tensor(train_X)))
train_pred = ProtoCloud.utils.process_prediction_file(
    train_pred, data.cell_encoder,
    label=data.cell_encoder.inverse_transform(train_Y))

# Compute and save per-class certainty thresholds
cls_threshold = ProtoCloud.utils.get_cls_threshold(train_pred)
ProtoCloud.utils.data_info_saver(cls_threshold, MODEL_DIR, 'cls_threshold')

train_pred.head(2)

,prob1,prob2,idx1,idx2,sim_proto,sim_score,certainty,certainty_threshold,ll_threshold,mis_pred,mis_anno,pred1,pred2,label
0,0.99988,0.000029,2,3,3,0.686477,certain,0.640877,None,False,False,CD4 T cells,CD8 T cells,CD4 T cells
1,0.99579,0.001042,5,7,3,0.676646,certain,0.673356,None,False,False,FCGR3A+ Monocytes,NK cells,FCGR3A+ Monocytes


### 4b. Test-set predictions

`get_predictions` returns a dict with keys:
- `sim_score` — similarity score to the nearest prototype
- `idx1` / `idx2` — top-1 / top-2 class indices
- `prob1` / `prob2` — softmax probabilities

In [10]:
predicted = pd.DataFrame(ProtoCloud.model.get_predictions(model, torch.Tensor(test_X)))
predicted = ProtoCloud.utils.process_prediction_file(
    predicted, data.cell_encoder,
    label=test_Y,
    model_dir=MODEL_DIR)
# New columns added: pred1, pred2, label, certainty

predicted.head()

,prob1,prob2,idx1,idx2,sim_proto,sim_score,certainty,certainty_threshold,ll_threshold,mis_pred,mis_anno,pred1,pred2,label
0,0.999960,1.394634e-05,1,5,5,0.818210,certain,0.655505,NaN,False,False,CD14+ Monocytes,FCGR3A+ Monocytes,CD14+ Monocytes
1,0.999661,8.562236e-05,3,0,2,0.728692,certain,0.668783,NaN,False,False,CD8 T cells,B cells,CD8 T cells
2,0.999698,7.587131e-05,3,4,5,0.775057,certain,0.668783,NaN,False,False,CD8 T cells,Dendritic Cells,CD8 T cells
3,0.999994,1.087916e-06,2,0,2,0.840945,certain,0.640877,NaN,False,False,CD4 T cells,B cells,CD4 T cells
4,0.999998,4.756019e-07,2,3,3,0.895896,certain,0.640877,NaN,False,False,CD4 T cells,CD8 T cells,CD4 T cells


### 4c. Similarity calibrator

`simCalibration` uses isotonic regression to convert raw similarity scores into calibrated probabilities.

In [11]:
# Fit calibrator on training predictions
calibrator = ProtoCloud.model.simCalibration()
calibrator.fit(
    similarity_score=train_pred['sim_score'].values,
    true_labels=train_pred['label'].values,
    pred_labels=train_pred['pred1'].values,
)
calibrator.save(MODEL_DIR)    # → calibrator_model.pkl

○ FCGR3A+ Monocytes: 576 samples - using global calibrator
○ NK cells: 576 samples - using global calibrator
○ Dendritic Cells: 576 samples - using global calibrator
○ Megakaryocytes: 576 samples - using global calibrator


In [13]:
# Load and apply calibrator at inference
calibrator = ProtoCloud.model.simCalibration.load(MODEL_DIR)
predicted['calibrated_certainty'] = calibrator.predict_proba(
    predicted['sim_score'].values,
    predicted['pred1'].values,
)

# Evaluate calibration quality (requires true labels)
metrics = calibrator.evaluate_calibration(
    predicted['sim_score'].values,
    predicted['calibrated_certainty'].values,
    test_Y,
    predicted['pred1'].values,
)
# Returns: brier_score_original, brier_score_calibrated, ece_original, ece_calibrated

Original Brier Score: 0.0765, ECE: 0.2158
Calibrated Brier Score: 0.0286, ECE: 0.0286
Brier Improvement: 0.04783354988486785
ECE Improvement: 0.18716138380305827


**Summary of key output columns:**

| Column | Description |
|--------|-------------|
| `pred1` | Top-1 predicted cell type |
| `pred2` | Top-2 predicted cell type |
| `prob1` / `prob2` | Softmax probabilities |
| `sim_score` | Raw prototype similarity score (0–1) |
| `certainty` | `certain` or `ambiguous` based on per-class threshold |
| `calibrated_certainty` | Isotonic-calibrated confidence (0–1) |
| `label` | True cell type label (if provided) |

---
## 5. Load & Apply a Pretrained Model

In [ ]:
# Load model architecture and weights
model2 = ProtoCloud.model.load_model(MODEL_DIR, exp_code=EXP_CODE, device=device)
cell_encoder = ProtoCloud.utils.data_info_loader('cell_encoder', MODEL_DIR)
model2.eval()
print('Model loaded successfully')

Model loaded
Model loaded successfully


To apply to a **new dataset** with a different gene set, use `gene_subset` to align genes:

In [ ]:
data_new = ProtoCloud.data.scRNAData(dataset_name='PBMC_10K_full', data_dir=DATA_DIR)
data_new.gene_subset(MODEL_DIR + EXP_CODE + '.pth')   # aligns genes to pretrained model

new_X = torch.Tensor(data_new.to_dense(data_new.adata))

Loading outside dataset, this will not process the data
AnnData object with n_obs × n_vars = 11527 × 3267
    obs: 'n_counts', 'batch', 'labels', 'str_labels', 'celltype'
    var: 'gene_symbols', 'n_counts-0', 'n_counts-1', 'n_counts', 'gene_name'
    uns: 'cell_types', 'log1p'
    obsm: 'design', 'normalized_qc', 'qc_pc', 'raw_qc'
    layers: 'counts'
Use genes as:  
load saved gene names from: ./tutorial/saved_models/PBMC_10K_full
number of genes in loaded model:  3267
3267 3267
	Shared genes in loaded model: 100.00%
	Shared genes in new dataset: 100.00%
3267
(11527, 3267) 3267 3267


C:\Users\yun\AppData\Roaming\Python\Python311\site-packages\scipy\sparse\_index.py:210: SparseEfficiencyWarning: Changing the sparsity structure of a csr_matrix is expensive. lil and dok are more efficient.
  self._set_arrayXarray(i, j, x)


(11527, 3267) 3267 3267
AnnData object with n_obs × n_vars = 11527 × 3267
    obs: 'n_counts', 'batch', 'labels', 'str_labels', 'celltype'
    var: 'gene_name'
    layers: 'counts'
	Using layer 'counts' as input


In [ ]:
predicted_new = pd.DataFrame(ProtoCloud.model.get_predictions(model2, new_X))
predicted_new = ProtoCloud.utils.process_prediction_file(predicted_new, cell_encoder,
                                                          model_dir=MODEL_DIR)
predicted_new.head(2)

,prob1,prob2,idx1,idx2,sim_proto,sim_score,certainty,certainty_threshold,ll_threshold,mis_pred,mis_anno,pred1,pred2,label
0,0.999982,0.000004,2,1,0,0.789712,certain,0.640877,NaN,None,None,CD4 T cells,CD14+ Monocytes,None
1,0.999974,0.000006,2,0,5,0.782067,certain,0.640877,NaN,None,None,CD4 T cells,B cells,None
2,0.999926,0.000024,1,3,5,0.830242,certain,0.655505,NaN,None,None,CD14+ Monocytes,CD8 T cells,None
3,0.999894,0.000026,1,4,1,0.784218,certain,0.655505,NaN,None,None,CD14+ Monocytes,Dendritic Cells,None
4,0.999551,0.000119,3,1,5,0.747603,certain,0.668783,NaN,None,None,CD8 T cells,CD14+ Monocytes,None


---
## 6. Continue Training on an Existing Model

Use this when you want to fine-tune a pretrained model or train for additional epochs.  
Always set `two_step=False` when continuing from a checkpoint.

In [24]:
# Load the existing model
# model2 = ProtoCloud.protoCloud(**model_dict_loaded).to(device)
# ProtoCloud.model.load_model(MODEL_DIR + EXP_CODE + '.pth', model2)
model2 = ProtoCloud.model.load_model(MODEL_DIR, exp_code=EXP_CODE, device=device)

# Continue training — always set two_step=False
ProtoCloud.model.run_model(
    model2, train_X, train_Y,
    epochs=10,
    two_step=False,     # IMPORTANT: False when continuing from a checkpoint
)

# Save the updated model
# ProtoCloud.utils.save_model(model2, MODEL_DIR, exp_code=EXP_CODE + '_cont')

Model loaded
loss coef: {'crs_ent': 1, 'recon': 10, 'kl': 2, 'ortho': 0.3, 'atomic': 1}
Start training
Train epoch: 0 	accu: 99.80030429821225% 	loss: 37.805465512159394 	recons: 3.7505346682013534 	KL: 0.2141100015582108 	cross ent: 0.0007288509439648606 	ortho: 1.3791273119972973 	atomic: -0.5425683274501707
Train epoch: 10 	accu: 99.809813617345% 	loss: 37.660506178693076 	recons: 3.7369246017642137 	KL: 0.2193763768527566 	cross ent: 0.0001923164971053555 	ortho: 1.3787258296478084 	atomic: -0.5613029642802912

Finished training
Total training time: 95.58 seconds
	Saving model to ./tutorial/saved_models/PBMC_10K_full/tutorial_run_cont.pth...
Model saved


> **Important:** Always use the **same `cell_encoder`** as the original model so that cell type integer indices remain consistent across training runs.

---
## 7. Using Predicted Labels as New Training Labels

This enables iterative self-training when the target dataset has no ground-truth labels.


In [ ]:
# Step 1 — Apply a pretrained model to the unlabeled target data (see §5)

# Step 2 — Use certain predicted cells as training set
# data.use_pred_label(results_dir, exp_code) replaces obs['celltype'] with predicted labels
train_idx, test_idx = data.get_split_idx(new_label=True, index_file='./my_split.csv')
train_X1, test_X1, train_Y1, test_Y1 = data.split_data(train_idx, test_idx)

# Continue training with pseudo-labels
ProtoCloud.model.run_model(
    model, train_X1, train_Y1,
    epochs=30,
    two_step=False,
)
# ProtoCloud.utils.save_model(model, <new_model_dir>, exp_code='run1_cont_train')


---
## 8. PRP Explanations (Gene Relevance)

**Prototype Relevance Propagation (PRP)** backpropagates prediction scores through the network to assign a relevance score to each gene for each prototype.

In [25]:
# Step 1 — Wrap the trained model with LRP rules
model_wrapped = ProtoCloud.protoCloud(**model_dict).to(device)
ProtoCloud.prp.convert_protocloud_to_lrp(
    model_wrapped, model,
    ProtoCloud.prp.lrp_params_def1,
    ProtoCloud.prp.rule_map,
)

# Step 2 — Generate PRP explanations
prp_path = './results/PBMC_10K_full/prp/'
ProtoCloud.utils.makedir(prp_path)

checkpoint = ProtoCloud.prp.generate_PRP_explanations(
    model_wrapped, train_X, train_Y,
    data,
    epsilon=model.epsilon,
    num_classes=num_classes,
    prototypes_per_class=6,
    prp_path=prp_path,
    exp_code=EXP_CODE,
    pretrain_model_pth=MODEL_DIR + EXP_CODE + '.pth',
)
np.save(MODEL_DIR + 'prototype_checkpoint.npy', checkpoint)


ProtoCloud LRP Model Conversion Complete.
Generating PRP explanations
Loading previous prototype states from ./tutorial/saved_models/PBMC_10K_full
Init relevance:  -1.0 1.0
sim_score rel backward:  -0.006070441100746393 3.4978911876678467
Class 0 Proto 0: Count = 11, min=0.000000, max=0.156801
Init relevance:  -1.0 1.0
sim_score rel backward:  -0.00603803526610136 3.7125656604766846
Class 0 Proto 1: Count = 425, min=0.000000, max=0.169046
Init relevance:  -1.0 1.0
sim_score rel backward:  -0.008187350817024708 3.678450107574463
Class 0 Proto 2: Count = 91, min=0.000000, max=0.180921
Init relevance:  -1.0 1.0
sim_score rel backward:  -0.005629689432680607 3.703766345977783
Class 0 Proto 3: Count = 499, min=0.000000, max=0.180740
Init relevance:  -1.0 1.0
sim_score rel backward:  -0.007747064344584942 3.6809401512145996
Class 0 Proto 4: Count = 2, min=0.000000, max=0.151832
Init relevance:  -1.0 1.0
sim_score rel backward:  -0.013753694482147694 3.63814640045166
Class 0 Proto 5: Count = 

C:\Users\yun\AppData\Roaming\Python\Python311\site-packages\anndata\_core\aligned_df.py:68: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


**Output files written to `prp_path`:**

| File | Contents |
|------|----------|
| `<celltype>_relgenes.npy` | Relevance matrix (n_prototypes × n_genes) for all prototypes of the celltype |
| `celltype_PRP.csv` | Weighted gene relevence per celltype |